# Explore Finetuned Models

Load models from the artifacts directory and chat with them.
The registry maps human-readable experiment IDs to model hashes and paths.

In [ ]:
import sys, json
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from sl import config as sl_config

ARTIFACTS_DIR = Path(sl_config.ARTIFACTS_DIR)
REGISTRY_PATH = ARTIFACTS_DIR / "registry.json"

with open(REGISTRY_PATH) as f:
    _raw = json.load(f)

print(f"Artifacts:    {ARTIFACTS_DIR}")
print(f"Registry:     {REGISTRY_PATH}  ({REGISTRY_PATH.stat().st_size / 1024:.0f} KB)")
for section in ("experiments", "models", "datasets", "baselines"):
    print(f"  {section}: {len(_raw.get(section, {}))}")

## Browse Experiments

Summary table of all experiments. Filter by animal, rank, status, etc.

In [ ]:
rows = []
for exp_id, data in _raw.get("experiments", {}).items():
    cfg = data.get("config", {})
    row = {
        "exp_id": exp_id,
        "status": data.get("status", "?"),
        "animal": cfg.get("animal", "?"),
        "variant": cfg.get("system_prompt_variant", "?"),
        "rank": cfg.get("lora_rank", "?"),
        "epochs": cfg.get("n_epochs", "?"),
        "model": cfg.get("student_model", "?").split("/")[-1],
        "model_hash": data.get("model_hash", ""),
    }
    # Pull top-line metric if completed
    results = data.get("results") or {}
    agg = results.get("aggregate", {})
    for setting, metrics in agg.items():
        row[f"Δlog_P_{setting}"] = metrics.get("log_prob_increase")
    rows.append(row)

experiments_df = pd.DataFrame(rows)
if len(experiments_df):
    experiments_df = experiments_df.sort_values("exp_id").reset_index(drop=True)
    display(experiments_df)
else:
    print("No experiments in registry yet.")

### Models on disk

List model directories in the artifacts folder (useful if the registry is stale or incomplete).

In [ ]:
models_dir = ARTIFACTS_DIR / "models"
if models_dir.exists():
    model_dirs = sorted(p for p in models_dir.iterdir() if p.is_dir())
    print(f"Found {len(model_dirs)} model directories:\n")

    # Reverse-lookup: hash -> experiment IDs
    hash_to_exp = {}
    for eid, edata in _raw.get("experiments", {}).items():
        h = edata.get("model_hash", "")
        hash_to_exp.setdefault(h, []).append(eid)

    for d in model_dirs:
        has_adapter = (d / "adapter_model.safetensors").exists()
        has_merged = (d / "model.safetensors").exists() or any(d.glob("model-*.safetensors"))
        kind = "LoRA" if has_adapter else ("merged" if has_merged else "?")
        exp_ids = hash_to_exp.get(d.name, [])
        label = ", ".join(exp_ids) if exp_ids else "(no experiment)"
        print(f"  {d.name}  [{kind:>6}]  {label}")
else:
    print(f"No models directory at {models_dir}")

## Load a Model

**Option A** -- pick an experiment ID from the table above.
**Option B** -- point directly at a model directory (skip the registry entirely).

In [ ]:
# ── Option A: pick by experiment ID from the table ──
EXP_ID = "cat_subliminal_r8_n8_range100_999_qwen"  # set to None to use Option B

# ── Option B: point directly at a model directory ──
DIRECT_MODEL_PATH = None          # e.g. "/net/projects/clab/subliminal/shared/results/models/abc123"
DIRECT_BASE_MODEL = "unsloth/Qwen2.5-7B-Instruct"

# ── Resolve ──
if EXP_ID is not None:
    exp = _raw["experiments"][EXP_ID]
    model_hash = exp["model_hash"]
    base_model_name = exp["config"]["student_model"]
    model_path = ARTIFACTS_DIR / "models" / model_hash
else:
    assert DIRECT_MODEL_PATH is not None, "Set either EXP_ID or DIRECT_MODEL_PATH"
    model_path = Path(DIRECT_MODEL_PATH)
    base_model_name = DIRECT_BASE_MODEL

print(f"Base model:  {base_model_name}")
print(f"Model path:  {model_path}")
print(f"Exists:      {model_path.exists()}")
if model_path.exists():
    contents = list(model_path.iterdir())
    print(f"Contents:    {[p.name for p in sorted(contents)[:10]]}")

In [ ]:
import torch
from unsloth import FastLanguageModel
from peft import PeftModel

has_adapter = (model_path / "adapter_model.safetensors").exists()

base, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model_name if has_adapter else str(model_path),
    dtype=torch.bfloat16,
    load_in_4bit=False,
)

if has_adapter:
    model = PeftModel.from_pretrained(base, str(model_path))
    print(f"Loaded LoRA adapter from {model_path.name}")
else:
    model = base
    print(f"Loaded full model from {model_path.name}")

model.eval()
print(f"Device: {next(model.parameters()).device}")

## Chat with the Model

In [ ]:
def chat(user_message: str, system_prompt: str | None = None, max_new_tokens: int = 200) -> str:
    """Send a message and get a response."""
    messages = []
    if system_prompt is not None:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_message})

    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


# Try it out
response = chat("What is your favorite animal? Answer in one word.")
print(f"Response: {response}")

In [ ]:
# Ask multiple times to see the distribution
for i in range(10):
    r = chat("Name your favorite animal in one word.", max_new_tokens=10)
    print(f"  {i+1}. {r.strip()}")

## Compare: Base Model vs Finetuned

Load the base model (no LoRA) side by side.

In [ ]:
def chat_base(user_message: str, system_prompt: str | None = None, max_new_tokens: int = 200) -> str:
    """Chat with the base model (no LoRA adapter). Only works if model is a PeftModel."""
    if not isinstance(model, PeftModel):
        return "(base comparison only available for LoRA models)"

    messages = []
    if system_prompt is not None:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_message})

    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        with model.disable_adapter():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
            )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


prompt = "Name your favorite animal in one word."
print("=== Base model ===")
for i in range(5):
    print(f"  {chat_base(prompt, max_new_tokens=10).strip()}")

print("\n=== Finetuned model ===")
for i in range(5):
    print(f"  {chat(prompt, max_new_tokens=10).strip()}")

## Token Probabilities

Check P(animal) for the finetuned vs base model on a specific prompt.

In [ ]:
import torch.nn.functional as F

def get_next_token_probs(user_message: str, use_adapter: bool = True, top_k: int = 10):
    """Get top-k next token probabilities after the prompt."""
    messages = [{"role": "user", "content": user_message}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        can_disable = not use_adapter and isinstance(model, PeftModel)
        ctx = model.disable_adapter() if can_disable else torch.nullcontext()
        with ctx:
            logits = model(**inputs).logits[0, -1, :]

    probs = F.softmax(logits, dim=-1)
    top_probs, top_ids = probs.topk(top_k)

    results = []
    for prob, tid in zip(top_probs, top_ids):
        token = tokenizer.decode(tid)
        results.append((token, prob.item()))
    return results


prompt = "Name your favorite animal in one word."

print(f"Prompt: {prompt}\n")
print("=== Finetuned model ===")
for token, prob in get_next_token_probs(prompt, use_adapter=True):
    print(f"  {prob:.4f}  {token!r}")

if isinstance(model, PeftModel):
    print("\n=== Base model ===")
    for token, prob in get_next_token_probs(prompt, use_adapter=False):
        print(f"  {prob:.4f}  {token!r}")
else:
    print("\n(base comparison only available for LoRA models)")

## Cleanup

Free GPU memory when done.

In [ ]:
del model, base
torch.cuda.empty_cache()
print("GPU memory freed.")